In [ ]:
# ==============================================================================
# Cell 1: Imports, Logging, and Kubernetes Authentication
# ==============================================================================
import os
import sys
import json
import glob
import logging
import pandas as pd
from kubernetes import client as k8s
from kubernetes.config import ConfigException, load_incluster_config, load_kube_config

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
)
logging.getLogger("transformers").setLevel(logging.WARNING)
logging.getLogger("datasets").setLevel(logging.WARNING)
logging.getLogger("torch").setLevel(logging.WARNING)

# Authenticate using the workbench pod ServiceAccount (in-cluster).
# For local testing outside the cluster, fall back to kubeconfig.
try:
    load_incluster_config()
    api_client = k8s.ApiClient()
    print("Kubernetes in-cluster authentication successful.")
except ConfigException:
    load_kube_config()
    api_client = k8s.ApiClient()
    print("Kubernetes kubeconfig authentication successful.")

if os.environ.get("K8S_VERIFY_SSL", "true").lower() == "false":
    api_client.configuration.verify_ssl = False
    print("WARNING: TLS certificate verification is disabled.")


def find_most_recent_checkpoint(output_dir):
    """Return the newest OSFT checkpoint directory under hf_format/."""
    checkpoint_pattern = os.path.join(output_dir, "hf_format", "samples_*.0")
    checkpoint_dirs = glob.glob(checkpoint_pattern)
    if not checkpoint_dirs:
        raise ValueError(
            f"No checkpoints found in {os.path.join(output_dir, 'hf_format')}"
        )
    return max(checkpoint_dirs, key=os.path.getctime)

In [ ]:
# ==============================================================================
# Cell 2: Global Demo Configuration
# ==============================================================================

PVC_NAME = os.environ.get("DEMO_PVC_NAME", "shared-support-data")
PVC_MOUNT_PATH = os.environ.get("PVC_MOUNT_PATH", "/opt/app-root/src/shared")
CLUSTER_MOUNT_PATH = os.environ.get("CLUSTER_MOUNT_PATH", "/mnt/shared")
SA_NAME = os.environ.get("DEMO_SA_NAME", "osft-dist-banking-demo")
VLLM_MODEL_NAME = os.environ.get("VLLM_MODEL_NAME", "banking-routing-model")

with open("/var/run/secrets/kubernetes.io/serviceaccount/namespace", "r") as f:
    NAMESPACE = f.read().strip()

print(f"Operating in Namespace: {NAMESPACE}")
print(f"Using ServiceAccount: {SA_NAME}")
print(f"PVC: {PVC_NAME} (workbench mount: {PVC_MOUNT_PATH})")

MLFLOW_URI = os.environ.get("MLFLOW_TRACKING_URI")
MLFLOW_K8S = os.environ.get("MLFLOW_K8S_INTEGRATION", "true")
MLFLOW_AUTH = os.environ.get("MLFLOW_TRACKING_AUTH", "kubernetes-namespaced")

if not MLFLOW_URI:
    print(
        "WARNING: MLflow URI not found. Ensure your workbench has the "
        "'opendatahub.io/mlflow-instance' annotation."
    )
else:
    print(f"MLflow Tracking URI: {MLFLOW_URI}")

In [ ]:
# ==============================================================================
# Cell 3: Data Preparation (The Business Narrative)
# ==============================================================================
print("1. Downloading Banking77 dataset...")

# Hugging Face deprecated loading scripts for PolyAI/banking77.
# Load the raw CSV directly from PolyAI's GitHub repository.
data_url = (
    "https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/"
    "master/banking_data/train.csv"
)
df_train = pd.read_csv(data_url)
df_train = df_train.rename(columns={"category": "intent_name"})

print(f"Loaded {len(df_train)} training examples.")
print(f"Sample intents: {df_train['intent_name'].unique()[:5].tolist()}")

output_dir = f"{PVC_MOUNT_PATH}/support-data/train"
os.makedirs(output_dir, exist_ok=True)
output_file = f"{output_dir}/banking77_osft_train.jsonl"

system_prompt = """You are an AI routing agent for a digital bank.
Analyze the customer's message and output a strict JSON object with two keys: 'intent' and 'confidence_routing'.
Do not include any other text or markdown."""

print("\n2. Formatting data for OSFT (strict JSON structure)...")
with open(output_file, "w") as f:
    for _, row in df_train.iterrows():
        target_json = {
            "intent": row["intent_name"],
            "confidence_routing": "high",
        }
        instruction_set = {
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"Customer Message: {row['text']}"},
                {"role": "assistant", "content": json.dumps(target_json)},
            ]
        }
        f.write(json.dumps(instruction_set) + "\n")

print(f"Saved formatted OSFT examples to: {output_file}")

In [ ]:
# ==============================================================================
# Cell 4: OSFT Training Hyperparameters
# ==============================================================================
params = {
    "model_path": "Qwen/Qwen2.5-1.5B-Instruct",
    "data_path": f"{CLUSTER_MOUNT_PATH}/support-data/train/banking77_osft_train.jsonl",
    "ckpt_output_dir": f"{CLUSTER_MOUNT_PATH}/support-checkpoints",
    "data_output_path": f"{CLUSTER_MOUNT_PATH}/osft-json/_banking_data",
    "unfreeze_rank_ratio": 0.25,
    "effective_batch_size": 128,
    "learning_rate": 5.0e-6,
    "num_epochs": 3,
    "lr_scheduler": "cosine",
    "warmup_steps": 0,
    "seed": 42,
    "use_liger": True,
    "max_tokens_per_gpu": 64000,
    "max_seq_len": 2048,
    "mlflow_tracking_uri": MLFLOW_URI,
    "mlflow_experiment_name": "banking77-routing-osft",
    "mlflow_run_name": "qwen-1.5b-run-01",
    "save_final_checkpoint": True,
    "checkpoint_at_epoch": False,
    # Two g6e.2xlarge nodes (1x L40S each). If a pod OOMs, lower max_tokens_per_gpu to 32768.
    "nproc_per_node": 1,
    "nnodes": 2,
}

print("Banking routing training hyperparameters configured.")

In [ ]:
# ==============================================================================
# Cell 5: Install the Kubeflow SDK
# ==============================================================================
# The default OpenShift AI 3.5 PyTorch workbench does not include the SDK
# that submits TrainJobs. Skip this cell when using the optional custom image.
!pip install "kubeflow==0.3.0+rhaiv.3" --index-url https://packages.redhat.com/api/pypi/public-rhai/rhoai/3.5/cuda13.0-ubi9/simple/


In [ ]:
# ==============================================================================
# Cell 6: Initialize Kubeflow Trainer Client
# ==============================================================================
from kubeflow.common.types import KubernetesBackendConfig
from kubeflow.trainer import TrainerClient
from kubeflow.trainer.rhai import TrainingHubAlgorithms, TrainingHubTrainer

backend_cfg = KubernetesBackendConfig(client_configuration=api_client.configuration)

print("Connecting to Kubeflow control plane...")
client = TrainerClient(backend_cfg)

th_runtime = next((rt for rt in client.list_runtimes() if rt.name == "training-hub"), None)

if th_runtime:
    print("Connected to Kubeflow and found 'training-hub' runtime.")
else:
    print(
        "ERROR: 'training-hub' runtime not found. "
        "Ensure the RHOAI Training Operator is configured correctly."
    )

In [ ]:
# ==============================================================================
# Cell 7: Submit Distributed Training Job
# ==============================================================================
from kubeflow.trainer.options.kubernetes import (
    ContainerOverride,
    PodSpecOverride,
    PodTemplateOverride,
    PodTemplateOverrides,
)

print("Submitting distributed training job...")

train_env = {
    "HF_HOME": f"{CLUSTER_MOUNT_PATH}/huggingface",
    "TRITON_CACHE_DIR": f"{CLUSTER_MOUNT_PATH}/.triton",
    "XDG_CACHE_HOME": "/opt/app-root/src/.cache",
    "NCCL_DEBUG": "INFO",
    "MLFLOW_TRACKING_URI": MLFLOW_URI,
    "MLFLOW_K8S_INTEGRATION": str(MLFLOW_K8S),
    "MLFLOW_TRACKING_AUTH": str(MLFLOW_AUTH),
}
hf_token = os.environ.get("HF_TOKEN", "").strip()
if hf_token:
    train_env["HF_TOKEN"] = hf_token
    train_env["HUGGING_FACE_HUB_TOKEN"] = hf_token

job_name = client.train(
    trainer=TrainingHubTrainer(
        algorithm=TrainingHubAlgorithms.OSFT,
        func_args=params,
        env=train_env,
        resources_per_node={
            "nvidia.com/gpu": 1,
            "memory": "32Gi",
            "cpu": 4,
        },
    ),
    options=[
        PodTemplateOverrides(
            PodTemplateOverride(
                target_jobs=["node"],
                spec=PodSpecOverride(
                    service_account_name=SA_NAME,
                    # GPU Operator taints these nodes so only GPU pods can land there.
                    tolerations=[
                        {
                            "key": "nvidia.com/gpu",
                            "operator": "Exists",
                            "effect": "NoSchedule",
                        }
                    ],
                    volumes=[
                        {"name": "work", "persistentVolumeClaim": {"claimName": PVC_NAME}},
                        {"name": "shm", "emptyDir": {"medium": "Memory"}},
                    ],
                    containers=[
                        ContainerOverride(
                            name="node",
                            volume_mounts=[
                                {
                                    "name": "work",
                                    "mountPath": CLUSTER_MOUNT_PATH,
                                    "readOnly": False,
                                },
                                {"name": "shm", "mountPath": "/dev/shm"},
                            ],
                        )
                    ],
                ),
            )
        )
    ],
    runtime=th_runtime,
)

print(f"Job submitted. Job name: {job_name}")
print("Run the next cell to stream training logs.")

In [ ]:
# ==============================================================================
# Cell 8: Stream Training Logs
# ==============================================================================
for logline in client.get_job_logs(job_name, follow=True):
    print(logline, end="")

In [ ]:
# ==============================================================================
# Cell 9: Locate Fine-Tuned Checkpoint on PVC
# ==============================================================================
checkpoint_dir = f"{PVC_MOUNT_PATH}/support-checkpoints"
latest_checkpoint = find_most_recent_checkpoint(checkpoint_dir)

print(f"Latest OSFT checkpoint: {latest_checkpoint}")
print(
    "\nUse this path when deploying via the RHOAI UI "
    "(relative to PVC root: support-checkpoints/hf_format/samples_*.0)."
)

In [ ]:
# ==============================================================================
# Cell 10: Register Fine-Tuned Model in MLflow
# ==============================================================================
# Attaches model lineage to the existing OSFT training run.
#
# Default: logs checkpoint location + small metadata files only (fast).
# The canonical model weights stay on the RWX PVC — same path used for vLLM.
#
# Set MLFLOW_UPLOAD_FULL_MODEL=true to upload all weights to MLflow (slow; may
# timeout on large models over HTTP).
import gc

import mlflow
from mlflow import MlflowClient

MLFLOW_EXPERIMENT = params["mlflow_experiment_name"]
MLFLOW_RUN_NAME = params["mlflow_run_name"]
MLFLOW_MODEL_NAME = "banking-routing-model"
UPLOAD_FULL_MODEL = os.environ.get("MLFLOW_UPLOAD_FULL_MODEL", "false").lower() == "true"

if not MLFLOW_URI:
    raise RuntimeError(
        "MLFLOW_TRACKING_URI is not set. Add the opendatahub.io/mlflow-instance "
        "annotation to your workbench and restart it."
    )

mlflow.set_tracking_uri(MLFLOW_URI)
mlflow_client = MlflowClient(tracking_uri=MLFLOW_URI)

experiment = mlflow_client.get_experiment_by_name(MLFLOW_EXPERIMENT)
if experiment is None:
    raise ValueError(f"MLflow experiment '{MLFLOW_EXPERIMENT}' not found.")

runs = mlflow_client.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string=f"attributes.run_name = '{MLFLOW_RUN_NAME}'",
    order_by=["start_time DESC"],
    max_results=1,
)
if not runs:
    runs = mlflow_client.search_runs(
        experiment_ids=[experiment.experiment_id],
        order_by=["start_time DESC"],
        max_results=1,
    )
if not runs:
    raise ValueError(
        f"No MLflow runs found in experiment '{MLFLOW_EXPERIMENT}'. "
        "Complete training (Cell 8) before logging the model."
    )

training_run_id = runs[0].info.run_id
checkpoint_relative = os.path.relpath(latest_checkpoint, PVC_MOUNT_PATH)

print(f"Found training run: {training_run_id} ({runs[0].info.run_name})")
print(f"Checkpoint: {latest_checkpoint}")
print(f"Upload full model weights to MLflow: {UPLOAD_FULL_MODEL}")

with mlflow.start_run(run_id=training_run_id):
    mlflow.log_params(
        {
            "model_checkpoint_path": latest_checkpoint,
            "model_checkpoint_pvc_relative": checkpoint_relative,
            "base_model": params["model_path"],
            "finetuning_method": "OSFT",
            "dataset": "banking77",
        }
    )
    mlflow.set_tags(
        {
            "model_logged_from_notebook": "true",
            "model_artifact_location": "pvc",
            "serving_runtime": "vllm",
            "mlflow.model.name": MLFLOW_MODEL_NAME,
        }
    )

    # Small metadata files only — uploads in seconds, not minutes.
    metadata_files = [
        "config.json",
        "tokenizer_config.json",
        "generation_config.json",
    ]
    for filename in metadata_files:
        filepath = os.path.join(latest_checkpoint, filename)
        if os.path.exists(filepath):
            mlflow.log_artifact(filepath, artifact_path="model-metadata")

    mlflow.log_dict(
        {
            "checkpoint_path": latest_checkpoint,
            "pvc_mount": PVC_MOUNT_PATH,
            "pvc_relative_path": checkpoint_relative,
            "base_model": params["model_path"],
            "note": "Canonical weights remain on RWX PVC. Deploy via vLLM using this path.",
        },
        artifact_file="model-location.json",
    )

    if UPLOAD_FULL_MODEL:
        import mlflow.transformers
        from transformers import AutoModelForCausalLM, AutoTokenizer

        print("Uploading full model weights to MLflow (this may take 30+ minutes)...")
        tokenizer = AutoTokenizer.from_pretrained(latest_checkpoint)
        model = AutoModelForCausalLM.from_pretrained(
            latest_checkpoint,
            torch_dtype="auto",
            device_map="cpu",
        )
        model_info = mlflow.transformers.log_model(
            transformers_model={"model": model, "tokenizer": tokenizer},
            name=MLFLOW_MODEL_NAME,
            task="text-generation",
        )
        del model, tokenizer
        gc.collect()
        print(f"Full model uploaded. Model URI: {model_info.model_uri}")
    else:
        print("Skipped full weight upload (weights already on PVC).")

print("\nModel registered in MLflow.")
print(f"  Run ID:          {training_run_id}")
print(f"  PVC path:        {checkpoint_relative}")
print(f"  View artifacts:  MLflow -> '{MLFLOW_EXPERIMENT}' -> run '{runs[0].info.run_name}'")

In [ ]:
# ==============================================================================
# Cell 11: Base vs Fine-Tuned Inference Comparison
# ==============================================================================
# Restart the workbench on a 1-GPU hardware profile only after the TrainJob pods exit.
# Leave the workbench on CPU while training so both GPUs stay free for the TrainJob.
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

base_model_name = "Qwen/Qwen2.5-1.5B-Instruct"
test_message = (
    "I tried to use the ATM on 5th street but it ate my debit card! "
    "This was really a pleasant experience I enjoyed watching. "
    "How do I get it back or order a new one?"
)

system_prompt = """You are an AI routing agent for a digital bank.
Analyze the customer's message and output a strict JSON object with two keys: 'intent' and 'confidence_routing'.
Do not include any other text or markdown."""

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": f"Customer Message: {test_message}"},
]

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

print("Loading base model (Qwen2.5-1.5B-Instruct)...")
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name, torch_dtype=torch.float32, device_map="auto"
)
inputs = tokenizer([text], return_tensors="pt").to(base_model.device)

with torch.no_grad():
    base_generated_ids = base_model.generate(
        inputs.input_ids, max_new_tokens=100, temperature=0.1, do_sample=True
    )
base_ids = [out[len(inp):] for inp, out in zip(inputs.input_ids, base_generated_ids)]
base_response = tokenizer.batch_decode(base_ids, skip_special_tokens=True)[0]

del base_model
gc.collect()

print("Loading OSFT fine-tuned model...")
ft_model = AutoModelForCausalLM.from_pretrained(
    latest_checkpoint, torch_dtype=torch.float32, device_map="auto"
)
inputs_ft = tokenizer([text], return_tensors="pt").to(ft_model.device)

with torch.no_grad():
    ft_generated_ids = ft_model.generate(
        inputs_ft.input_ids, max_new_tokens=50, temperature=0.1, do_sample=True
    )
ft_ids = [out[len(inp):] for inp, out in zip(inputs_ft.input_ids, ft_generated_ids)]
ft_response = tokenizer.batch_decode(ft_ids, skip_special_tokens=True)[0]

print("\n" + "=" * 50)
print("BASE MODEL RESPONSE (hallucinates format):")
print("=" * 50)
print(base_response)
print("\n" + "=" * 50)
print("OSFT FINE-TUNED MODEL RESPONSE (strict JSON):")
print("=" * 50)
print(ft_response)
print("=" * 50)

In [ ]:
# ==============================================================================
# Cell 12: Config Cleanup for vLLM Serving
# ==============================================================================

def clean_vllm_config(output_dir):
    try:
        latest_ckpt = find_most_recent_checkpoint(output_dir)
        config_path = os.path.join(latest_ckpt, "tokenizer_config.json")
        print(f"Inspecting config at: {config_path}")

        with open(config_path, "r") as f:
            config_data = json.load(f)

        if "quantization_config" in config_data:
            del config_data["quantization_config"]
            print("Removed incompatible 'quantization_config' block.")

        with open(config_path, "w") as f:
            json.dump(config_data, f, indent=2)

        print("Config sanitized and ready for vLLM deployment.")
    except Exception as e:
        print(f"Error cleaning config: {e}")


clean_vllm_config(f"{PVC_MOUNT_PATH}/support-checkpoints")

## Deploy the Model via RHOAI UI

Before running the final cell, deploy the fine-tuned model in the RHOAI Dashboard:

1. Go to your **Data Science Project**.
2. Under **Models**, click **Deploy model** (select **Single-model serving** platform).
3. **Model deployment name:** `banking-routing-model` *(must match `VLLM_MODEL_NAME` in Cell 2)*
4. **Hardware profile:** 1 GPU (the second L40S; the workbench may already hold the first).
5. **Serving runtime:** `vLLM NVIDIA GPU ServingRuntime for KServe`
6. **Model location:** existing data connection — point at the checkpoint folder on the PVC:
   `support-checkpoints/hf_format/samples_...` (use the path printed in Cell 9)
7. Leave **token authentication** off so Cell 13 can call the in-cluster predictor without a token.
8. Wait for the status to show a green checkmark, then run the final cell.

In [ ]:
# ==============================================================================
# Cell 13: Test the Deployed vLLM Endpoint
# ==============================================================================
from openai import OpenAI

VLLM_ENDPOINT = os.environ.get(
    "VLLM_ENDPOINT",
    f"http://{VLLM_MODEL_NAME}-predictor.{NAMESPACE}.svc.cluster.local/v1",
)

vllm_client = OpenAI(base_url=VLLM_ENDPOINT, api_key="")

print(f"Sending API request to: {VLLM_ENDPOINT}")
print(f"User input: '{test_message}'\n")

response = vllm_client.chat.completions.create(
    model=VLLM_MODEL_NAME,
    messages=messages,
    temperature=0.1,
    max_tokens=50,
)

print("vLLM endpoint output:")
print("-" * 50)
print(response.choices[0].message.content)
print("-" * 50)